# Retrieval and terminology analysis

This notebook evaluates whether the DiReCT guideline retriever returns the gold diagnostic path. Lexical retrieval is the primary configuration; local UMLS normalization is included when a terminology index is available. All exported artifacts are aggregate and contain no clinical note text.

In [ ]:
from pathlib import Path
import os
import sys
import time

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "clinical_cds").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / "output/.matplotlib"))

import matplotlib.pyplot as plt
import pandas as pd

from clinical_cds.direct import load_direct_dataset
from clinical_cds.evaluation import normalized_label_key
from clinical_cds.medqa import load_medqa_cases
from clinical_cds.normalization import UMLSNormalizer
from clinical_cds.retrieval import KnowledgeRetriever

In [ ]:
DIRECT_ROOT = Path(os.environ.get("DIRECT_ROOT", REPO_ROOT / "data/mimic_iv_ext_direct/unpacked"))
MEDQA_ROOT = Path(os.environ.get("MEDQA_ROOT", REPO_ROOT / "data/medqa"))
OUTPUT_DIR = REPO_ROOT / "output/notebook_artifacts/retrieval_umls"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOP_K_VALUES = tuple(
    sorted({int(value) for value in os.environ.get("RETRIEVAL_TOP_K", "1,3,6,9,12").split(",")})
)

In [ ]:
dataset = load_direct_dataset(DIRECT_ROOT)
retrieval_configs = {"lexical": (KnowledgeRetriever(dataset.graphs), None)}

umls_value = os.environ.get("UMLS_DB")
default_umls_path = REPO_ROOT / "output/cache/umls_local.sqlite3"
umls_path = Path(umls_value) if umls_value else default_umls_path
if not umls_path.is_absolute():
    umls_path = REPO_ROOT / umls_path
if umls_path.is_file():
    umls_normalizer = UMLSNormalizer.from_path(umls_path)
    retrieval_configs["umls"] = (
        KnowledgeRetriever(dataset.graphs, normalizer=umls_normalizer),
        umls_normalizer,
    )
else:
    umls_normalizer = None
    print(f"UMLS sensitivity analysis skipped: no index at {umls_path}")

pd.DataFrame(
    [
        {
            "configuration": name,
            "retriever_id": retriever.retriever_id,
            "case_count": len(dataset.cases),
        }
        for name, (retriever, _) in retrieval_configs.items()
    ]
)

## Gold-path coverage by retrieval depth

A hit occurs when any returned fact names the gold diagnosis or places it on the returned diagnostic path. The graph-consistent subset excludes cases with missing graphs, out-of-graph conclusions, or non-leaf conclusions.

In [ ]:
graph_consistency_flags = {
    "missing_guideline_graph",
    "conclusion_outside_category_graph",
    "conclusion_not_leaf",
}
coverage_rows = []
started = time.perf_counter()

for configuration, (retriever, normalizer) in retrieval_configs.items():
    configuration_started = time.perf_counter()
    for case in dataset.cases:
        gold_key = normalized_label_key(case.gold_label, normalizer)
        graph_consistent = not bool(set(case.quality_flags) & graph_consistency_flags)
        for top_k in TOP_K_VALUES:
            bundle = retriever.retrieve(case, top_k=top_k)
            retrieved_keys = {
                normalized_label_key(label, normalizer)
                for fact in bundle.facts
                for label in (fact.diagnosis_label, *fact.diagnostic_path)
            }
            coverage_rows.append(
                {
                    "configuration": configuration,
                    "case_id": case.case_id,
                    "disease_category": case.disease_category,
                    "top_k": top_k,
                    "gold_path_retrieved": float(gold_key in retrieved_keys),
                    "graph_consistent": graph_consistent,
                }
            )
    print(
        f"{configuration}: {time.perf_counter() - configuration_started:.1f} seconds"
    )

coverage_cases = pd.DataFrame(coverage_rows)
coverage_summary_rows = []
for subset, subset_rows in (
    ("all", coverage_cases),
    ("graph_consistent", coverage_cases[coverage_cases["graph_consistent"]]),
):
    grouped = (
        subset_rows.groupby(["configuration", "top_k"], as_index=False)
        .agg(
            n=("case_id", "count"),
            gold_path_coverage=("gold_path_retrieved", "mean"),
        )
        .assign(subset=subset)
    )
    coverage_summary_rows.append(grouped)

coverage_summary = pd.concat(coverage_summary_rows, ignore_index=True)
coverage_summary = coverage_summary[
    ["subset", "configuration", "top_k", "n", "gold_path_coverage"]
]
coverage_summary.to_csv(OUTPUT_DIR / "retrieval_coverage_by_k.csv", index=False)
print(f"Total analysis time: {time.perf_counter() - started:.1f} seconds")
coverage_summary

In [ ]:
figure, axis = plt.subplots(figsize=(8.5, 5.0))
line_styles = {"all": "-", "graph_consistent": "--"}
colors = {"lexical": "#3478A5", "umls": "#D17A22"}
for (subset, configuration), rows in coverage_summary.groupby(
    ["subset", "configuration"], sort=False
):
    rows = rows.sort_values("top_k")
    axis.plot(
        rows["top_k"],
        rows["gold_path_coverage"],
        marker="o",
        linestyle=line_styles[subset],
        color=colors.get(configuration, "#687078"),
        label=f"{configuration}, {subset.replace('_', ' ')}",
    )
axis.set_xticks(TOP_K_VALUES)
axis.set_ylim(0, 1)
axis.set_xlabel("Retrieved guideline facts (top-k)")
axis.set_ylabel("Gold diagnostic path coverage")
axis.set_title("DiReCT retrieval coverage")
axis.grid(color="#D8D8D8", linewidth=0.6)
axis.set_axisbelow(True)
axis.legend(frameon=False)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "retrieval_coverage_by_k.png", dpi=220)
plt.show()

## Disease-family coverage at the study retrieval depth

In [ ]:
study_top_k = 6 if 6 in TOP_K_VALUES else max(TOP_K_VALUES)
category_coverage = (
    coverage_cases[coverage_cases["top_k"] == study_top_k]
    .groupby(["configuration", "disease_category"], as_index=False)
    .agg(
        n=("case_id", "count"),
        gold_path_coverage=("gold_path_retrieved", "mean"),
        graph_consistent_fraction=("graph_consistent", "mean"),
    )
)
category_coverage.to_csv(
    OUTPUT_DIR / f"retrieval_coverage_by_disease_top_{study_top_k}.csv",
    index=False,
)
category_coverage

## UMLS mapping coverage of guideline diagnoses

In [ ]:
if umls_normalizer is not None:
    mapping_rows = []
    for graph in dataset.graphs:
        labels = tuple(sorted(set(graph.diagnosis_labels)))
        mapped = sum(
            umls_normalizer.diagnosis_key(label).startswith("cui:")
            for label in labels
        )
        mapping_rows.append(
            {
                "disease_category": graph.category,
                "diagnosis_label_count": len(labels),
                "umls_mapped_count": mapped,
                "umls_mapping_fraction": mapped / len(labels) if labels else 0.0,
            }
        )
    umls_mapping = pd.DataFrame(mapping_rows)
else:
    umls_mapping = pd.DataFrame(
        columns=[
            "disease_category",
            "diagnosis_label_count",
            "umls_mapped_count",
            "umls_mapping_fraction",
        ]
    )

umls_mapping.to_csv(OUTPUT_DIR / "umls_guideline_mapping.csv", index=False)
umls_mapping

## MedQA diagnostic-question and graph coverage

This is a label-coverage audit, not a model evaluation. It quantifies how many diagnosis questions can be linked to the available DiReCT guideline inventory.

In [ ]:
medqa_rows = []
if MEDQA_ROOT.exists():
    for configuration, (_, normalizer) in retrieval_configs.items():
        for split in ("dev", "test"):
            try:
                cases = load_medqa_cases(
                    MEDQA_ROOT,
                    split=split,
                    graphs=dataset.graphs,
                    diagnostic_only=True,
                    normalizer=normalizer,
                )
            except FileNotFoundError as error:
                print(error)
                continue
            covered = sum(
                bool(case.metadata.get("direct_graph_covered")) for case in cases
            )
            medqa_rows.append(
                {
                    "configuration": configuration,
                    "split": split,
                    "diagnostic_question_count": len(cases),
                    "graph_covered_count": covered,
                    "graph_covered_fraction": covered / len(cases) if cases else 0.0,
                }
            )
else:
    print(f"MedQA audit skipped: no dataset at {MEDQA_ROOT}")

medqa_coverage = pd.DataFrame(
    medqa_rows,
    columns=[
        "configuration",
        "split",
        "diagnostic_question_count",
        "graph_covered_count",
        "graph_covered_fraction",
    ],
)
medqa_coverage.to_csv(OUTPUT_DIR / "medqa_graph_coverage.csv", index=False)
medqa_coverage

Retrieval coverage is a prerequisite measure rather than evidence of diagnostic correctness. Diagnostic outcomes and paired condition effects are reported in notebook `03_ablation_results.ipynb`.